# 🔬 Topological Churn Analysis — Cell2Cell Telecom
### Geometric Deep Learning + Semantic Causal Inference
**Pipeline**: `KeplerMapper → Topological Sampling → LLM Causal Discovery (Stage 3) → DoWhy (Stage 4)`

---

## 📥 Step 0: Get the Data

The **Cell2Cell dataset** comes from Duke University's Fuqua School of Business.  
It has **71,047 rows and 58 features** — call quality, roaming, customer service contacts, revenue, handset info, demographics.

### Option A — Direct from Duke (free, no login)
```
https://www2.stat.duke.edu/~banks/datasets/Cell2Cell.zip
```
Unzip → you'll get `cell2cell.csv` (some versions named `Cell2Cell_data.csv`).

### Option B — Kaggle (same data, easier UI)
```
https://www.kaggle.com/datasets/jpacse/datasets-for-churn-telecom
```
Download `cell2celltrain.csv` and `cell2cellholdout.csv`.  
We use `cell2celltrain.csv` (51,047 rows with churn labels) in this notebook.

### After downloading
Put the CSV in the **same folder as this notebook** and set `DATA_PATH` in the cell below.  
Both Duke and Kaggle versions are handled automatically.

---

## ⚙️ Install Dependencies

In [1]:
# Path fix + all imports are consolidated in the next cell. Skip this one.

## 📚 Imports

In [2]:
# ── Step 1: Add user site-packages to sys.path BEFORE any imports ─────────────
# Anaconda blocks user site-packages by default, so kmapper (installed there)
# is invisible even though pip says it's installed. This one-time fix adds it.
import sys, site
user_site = site.getusersitepackages()  # e.g. C:\Users\sanat\AppData\Roaming\Python\Python313\site-packages
if user_site not in sys.path:
    sys.path.insert(0, user_site)
print(f'sys.path fix applied: {user_site}')

# ── Step 2: All other imports ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import kmapper as km
from kmapper import Cover

from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

try:
    from umap import UMAP
    HAS_UMAP = True
    print('✅ UMAP available — will use as primary lens')
except ImportError:
    HAS_UMAP = False
    print('⚠️  UMAP not found — falling back to PCA lens')

import networkx as nx
import plotly.graph_objects as go
import plotly.express as px

np.random.seed(42)
print('✅ All imports OK')

sys.path fix applied: C:\Users\sanat\AppData\Roaming\Python\Python313\site-packages
✅ UMAP available — will use as primary lens
✅ All imports OK


---
## 📂 Stage 0: Load & Inspect Cell2Cell

In [3]:
# ── SET THIS PATH ──────────────────────────────────────────────────────────────
DATA_PATH = "cell2celltrain.csv"   # ← change to your filename if different
# Also accepted: "cell2cell.csv", "Cell2Cell_data.csv"
# ──────────────────────────────────────────────────────────────────────────────

raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
raw.head(3)

Loaded: 51,047 rows × 58 columns


,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,...,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,...,0,4,No,0,30,Yes,1-Highest,Suburban,Professional,No
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,...,0,5,No,0,30,No,4-Medium,Suburban,Professional,Yes
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,...,0,6,No,0,Unknown,No,3-Good,Town,Crafts,Yes


In [4]:
# ── Detect churn column (name varies by source) ────────────────────────────────
churn_col_candidates = [c for c in raw.columns if 'churn' in c.lower()]
print(f"Churn column candidates: {churn_col_candidates}")

# Standardise to lowercase column names
raw.columns = raw.columns.str.strip().str.lower().str.replace(' ', '_')

# Cell2Cell churn is stored as 'Yes'/'No' or 1/0 depending on source
CHURN_COL = [c for c in raw.columns if 'churn' in c][0]
print(f"Using churn column: '{CHURN_COL}'")
print(f"Churn values sample: {raw[CHURN_COL].value_counts().to_dict()}")

# Binarize if needed
if raw[CHURN_COL].dtype == object:
    raw[CHURN_COL] = raw[CHURN_COL].str.strip().str.upper().map({'YES': 1, 'NO': 0, 'TRUE': 1, 'FALSE': 0})

print(f"\nOverall churn rate: {raw[CHURN_COL].mean():.1%}")
print(f"Churners: {raw[CHURN_COL].sum():,} | Non-churners: {(raw[CHURN_COL] == 0).sum():,}")

Churn column candidates: ['Churn']
Using churn column: 'churn'
Churn values sample: {'No': 36336, 'Yes': 14711}

Overall churn rate: 28.8%
Churners: 14,711 | Non-churners: 36,336


In [5]:
# ── Column inventory ────────────────────────────────────────────────────────────
col_info = pd.DataFrame({
    'dtype':    raw.dtypes,
    'nulls':    raw.isnull().sum(),
    'null_pct': (raw.isnull().sum() / len(raw) * 100).round(1),
    'nunique':  raw.nunique(),
    'sample':   raw.iloc[0]
})
print("Full column inventory:")
print(col_info.to_string())

Full column inventory:
                             dtype  nulls  null_pct  nunique        sample
customerid                   int64      0       0.0    51047       3000002
churn                        int64      0       0.0        2             1
monthlyrevenue             float64    156       0.3    12665          24.0
monthlyminutes             float64    156       0.3     2719         219.0
totalrecurringcharge       float64    156       0.3      214          22.0
directorassistedcalls      float64    156       0.3      146          0.25
overageminutes             float64    156       0.3      746           0.0
roamingcalls               float64    156       0.3      530           0.0
percchangeminutes          float64    367       0.7     2262        -157.0
percchangerevenues         float64    367       0.7     2899         -19.0
droppedcalls               float64      0       0.0      285           0.7
blockedcalls               float64      0       0.0      373           0.7
un

---
## 🧹 Stage 0B: Preprocessing

Cell2Cell mixes numeric, binary (Yes/No), and categorical (PrizmCode, Occupation, CreditRating) features.  
We need to handle each type correctly before feeding into the Mapper.

In [6]:
df = raw.copy()

# ── The 18 core numeric behavioral features ────────────────────────────────────
# These are the most topologically informative (behavioral signals, not demographics)
NUMERIC_BEHAVIORAL = [
    'monthlyrevenue',       # Revenue per month
    'monthlyminutes',       # Call minutes per month
    'totalrecurringcharge', # Recurring charge
    'directorassistedcalls',# Calls needing directory assistance (friction signal)
    'overageminutes',       # Minutes over plan limit (pricing friction)
    'roamingcalls',         # Roaming calls
    'percchangeminutes',    # % change in call minutes MoM
    'percchangerevenues',   # % change in revenue MoM
    'droppedcalls',         # Dropped calls (network quality)
    'blockedcalls',         # Blocked calls (network quality)
    'unansweredcalls',      # Unanswered calls
    'customercarecalls',    # Customer care contacts (friction/dissatisfaction)
    'threewaycalls',        # Three-way calls
    'receivedcalls',        # Received calls
    'outboundcalls',        # Outbound calls
    'peakcallsinout',       # Peak hour calls in+out
    'offpeakcallsinout',    # Off-peak calls in+out
    'droppedblockdcalls',   # Combined dropped+blocked
    'callforwardingcalls',  # Call forwarding usage
    'callwaitingcalls',     # Call waiting usage
    'monthsinservice',      # Tenure in months
    'uniquesubs',           # Unique subscribers on account
    'activesubs',           # Active subscribers on account
    'handsets',             # Number of handsets owned
    'handsetmodels',        # Number of handset models
    'currentequipmentdays', # Days with current equipment
    'agehh1',               # Age of head of household 1
    'agehh2',               # Age of head of household 2
]

# Keep only columns that exist in this version of the dataset
NUMERIC_BEHAVIORAL = [c for c in NUMERIC_BEHAVIORAL if c in df.columns]
print(f"Numeric behavioral features found: {len(NUMERIC_BEHAVIORAL)}")
print(NUMERIC_BEHAVIORAL)

# ── Binary Yes/No features ─────────────────────────────────────────────────────
binary_candidates = [
    'childreninhh', 'handsetrefurbished', 'handsetwebcapable', 'truckowner',
    'rvowner', 'homeownership', 'buysviamailorder', 'respondstomailoffers',
    'optoutmailings', 'nonustravel', 'ownscomputer', 'hascreditcard',
    'retentioncall', 'retentionoffer', 'newcellphoneuser', 'notnewinlastyear',
    'madecalltoretentionteam'
]
BINARY_COLS = [c for c in binary_candidates if c in df.columns]
print(f"\nBinary Yes/No features found: {len(BINARY_COLS)}")

# ── Encode binary columns ──────────────────────────────────────────────────────
for col in BINARY_COLS:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.upper().map({'YES': 1, 'NO': 0})

# ── Encode ordinal categorical: CreditRating ──────────────────────────────────
CREDIT_COL = 'creditrating'
if CREDIT_COL in df.columns and df[CREDIT_COL].dtype == object:
    credit_order = {'poor': 1, 'fair': 2, 'good': 3, 'high': 4, 'medium': 3,
                    'average': 2, 'above average': 3, 'below average': 1}
    df[CREDIT_COL] = df[CREDIT_COL].str.strip().str.lower().map(credit_order)
    BINARY_COLS.append(CREDIT_COL)

# ── All model features ─────────────────────────────────────────────────────────
FEATURE_COLS = NUMERIC_BEHAVIORAL + BINARY_COLS
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f"\nTotal features for model: {len(FEATURE_COLS)}")

Numeric behavioral features found: 27
['monthlyrevenue', 'monthlyminutes', 'totalrecurringcharge', 'directorassistedcalls', 'overageminutes', 'roamingcalls', 'percchangeminutes', 'percchangerevenues', 'droppedcalls', 'blockedcalls', 'unansweredcalls', 'customercarecalls', 'threewaycalls', 'receivedcalls', 'outboundcalls', 'peakcallsinout', 'offpeakcallsinout', 'callforwardingcalls', 'callwaitingcalls', 'monthsinservice', 'uniquesubs', 'activesubs', 'handsets', 'handsetmodels', 'currentequipmentdays', 'agehh1', 'agehh2']

Binary Yes/No features found: 14

Total features for model: 42


In [7]:
X_raw = df[FEATURE_COLS].copy()
y     = df[CHURN_COL].copy()

# Drop any columns that are non-numeric — SimpleImputer silently removes them
# which causes a shape mismatch when rebuilding the DataFrame
non_numeric = X_raw.select_dtypes(exclude='number').columns.tolist()
if non_numeric:
    print(f"⚠️  Dropping non-numeric columns from FEATURE_COLS: {non_numeric}")
    X_raw = X_raw.drop(columns=non_numeric)
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in non_numeric]

print(f"Missing values before imputation: {X_raw.isnull().sum().sum():,}")
print(f"Feature count: {len(FEATURE_COLS)}")

imputer  = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=FEATURE_COLS,   # now guaranteed to match imputer output
    index=X_raw.index
)

print(f"Missing values after imputation: {X_imputed.isnull().sum().sum()}")

scaler    = StandardScaler()
X_scaled  = pd.DataFrame(scaler.fit_transform(X_imputed), columns=FEATURE_COLS)

print(f"\nFinal feature matrix: {X_scaled.shape[0]:,} rows × {X_scaled.shape[1]} features")
print(f"Churn rate: {y.mean():.1%}")

Missing values before imputation: 105,585
Feature count: 42


ValueError: Shape of passed values is (51047, 40), indices imply (51047, 42)

---
## 📊 Stage 0C: Why Global Models Fail on Cell2Cell

In [ ]:
# Train a global GBM — this is our baseline 'naive' model
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

gbm_global = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42, subsample=0.8)
gbm_global.fit(X_tr, y_tr)
global_auc = roc_auc_score(y_te, gbm_global.predict_proba(X_te)[:, 1])
print(f"Global GBM AUC: {global_auc:.4f}")

# Compute churn probability for each row — used as lens later
churn_proba = gbm_global.predict_proba(X_scaled)[:, 1]

# Feature importance
fi = pd.Series(gbm_global.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_fi = fi.head(20).sort_values()
colors = ['#e74c3c' if i >= len(top_fi) - 5 else '#3498db' for i in range(len(top_fi))]
top_fi.plot(kind='barh', ax=ax, color=colors)
ax.set_title(f'Global GBM Feature Importance (AUC = {global_auc:.3f})\n'
             f'These rankings HIDE heterogeneous causal mechanisms across subgroups',
             fontweight='bold')
ax.set_xlabel('Feature Importance')
plt.tight_layout()
plt.savefig('global_importance_cell2cell.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 5 globally important features: {list(fi.head(5).index)}")
print("→ TDA will show these rankings are DIFFERENT in each behavioral cluster")

---
## 🗺️ Stage 1: Topological Mapping with KeplerMapper

### Lens Strategy for Cell2Cell (58 features)

With 58 features, a strong lens is critical. We use a **2D lens**:

| Lens Dimension | Method | Captures |
|---|---|---|
| Dim 1 | Churn probability (from GBM) | The *output* direction — who is leaving |
| Dim 2 | UMAP component 1 (or PCA-1) | The dominant *behavioral* variation across customers |

This ensures the Mapper graph is organized both by **likelihood of churn** and **type of behavior**.

In [ ]:
# ── Build the 2D Lens ─────────────────────────────────────────────────────────
print("Building lens...")

if HAS_UMAP:
    print("  Running UMAP (n_neighbors=30, min_dist=0.1)...")
    reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                  random_state=42, n_jobs=-1)
    umap_embedding = reducer.fit_transform(X_scaled)
    structural_lens = umap_embedding[:, 0]
    print(f"  UMAP done. Embedding shape: {umap_embedding.shape}")
else:
    print("  Running PCA...")
    pca = PCA(n_components=2)
    pca_emb = pca.fit_transform(X_scaled)
    structural_lens = pca_emb[:, 0]
    print(f"  PCA explained variance: {pca.explained_variance_ratio_[:2]}")

# Combined 2D lens
lens_2d = np.column_stack([churn_proba, structural_lens])
lens_2d_scaled = MinMaxScaler().fit_transform(lens_2d)

print(f"\nLens shape: {lens_2d_scaled.shape}")
print("Lens Dim 0: Churn Probability")
print(f"Lens Dim 1: {'UMAP-1' if HAS_UMAP else 'PC-1'} (behavioral structure)")

In [ ]:
# ── Build the Mapper Graph ────────────────────────────────────────────────────
# Parameter rationale for Cell2Cell (71K rows, 58 features):
#   n_cubes=20  → finer resolution than default (15) due to high dimensionality
#   perc_overlap=0.4 → moderate overlap; keeps graph connected without over-merging
#   DBSCAN eps=0.6 → slightly larger radius for denser data

mapper = km.KeplerMapper(verbose=1)

cover = Cover(n_cubes=20, perc_overlap=0.4)

# For 71K rows, DBSCAN min_samples=5 prevents noise nodes
clusterer = DBSCAN(eps=0.6, min_samples=5, n_jobs=-1)

graph = mapper.map(
    lens_2d_scaled,
    X_scaled.values,
    cover=cover,
    clusterer=clusterer
)

n_nodes = len(graph['nodes'])
n_edges = sum(len(v) for v in graph['links'].values()) // 2
print(f"\n✅ Mapper graph built:")
print(f"   Nodes (customer clusters): {n_nodes}")
print(f"   Edges (transitions):       {n_edges}")

In [ ]:
# ── Annotate nodes ────────────────────────────────────────────────────────────
node_stats = {}
for node_id, member_indices in graph['nodes'].items():
    members_df  = df.iloc[member_indices]
    members_X   = X_imputed.iloc[member_indices]   # unscaled, for readable means

    stats = {
        'size':        len(member_indices),
        'churn_rate':  members_df[CHURN_COL].mean(),
        'churn_proba': churn_proba[member_indices].mean(),
        'n_churners':  int(members_df[CHURN_COL].sum()),
        'members':     member_indices,
    }
    # Mean of key causal features (raw scale for interpretability)
    for feat in NUMERIC_BEHAVIORAL[:15]:   # Top 15 behavioral features
        if feat in members_X.columns:
            stats[f'mean_{feat}'] = members_X[feat].mean()
    node_stats[node_id] = stats

stats_df = pd.DataFrame(node_stats).T.infer_objects()
for col in ['size', 'n_churners', 'churn_rate', 'churn_proba']:
    stats_df[col] = pd.to_numeric(stats_df[col], errors='coerce')

print(f"Node stats computed for {len(stats_df)} nodes")
print("\nDistribution of node churn rates:")
print(stats_df['churn_rate'].describe().round(3))

In [ ]:
# ── Build NetworkX graph ──────────────────────────────────────────────────────
G = nx.Graph()

for node_id, stats in node_stats.items():
    G.add_node(node_id, **{k: v for k, v in stats.items() if k != 'members'})

for node_id, neighbors in graph['links'].items():
    for nbr in neighbors:
        if not G.has_edge(node_id, nbr):
            set_a = set(graph['nodes'][node_id])
            set_b = set(graph['nodes'][nbr])
            jaccard = len(set_a & set_b) / len(set_a | set_b)
            G.add_edge(node_id, nbr, weight=jaccard)

components = list(nx.connected_components(G))
print(f"NetworkX: {G.number_of_nodes()} nodes | {G.number_of_edges()} edges")
print(f"Connected components: {len(components)}")  
print(f"Largest component: {max(len(c) for c in components)} nodes")

---
## 🔥 Stage 1D: Churn Flare Detection

In [ ]:
CHURN_FLARE_THRESHOLD = 0.45   # Nodes with >= 45% churn rate (well above 28% baseline)

degrees = dict(G.degree())

# Score: high churn + low connectivity = extreme flare tip
flare_scores = {}
for node in G.nodes():
    cr  = G.nodes[node].get('churn_rate', 0)
    deg = degrees[node]
    if cr >= CHURN_FLARE_THRESHOLD:
        flare_scores[node] = cr / (deg + 1)   # higher = more isolated flare tip

flare_sorted = sorted(flare_scores.items(), key=lambda x: -x[1])

print(f"High-churn nodes (churn rate >= {CHURN_FLARE_THRESHOLD:.0%}): {len(flare_scores)}")
print()
print(f"{'Node':<30} {'Churn Rate':>12} {'Customers':>10} {'Churners':>10} {'Degree':>8} {'Score':>10}")
print("-" * 82)
for node, score in flare_sorted[:12]:
    nd = G.nodes[node]
    print(f"{node:<30} {nd['churn_rate']:>12.1%} {int(nd['size']):>10,} {nd['n_churners']:>10,} "
          f"{degrees[node]:>8} {score:>10.3f}")

In [ ]:
# ── Profile the top flare ─────────────────────────────────────────────────────
TOP_N_FLARE_NODES = 6
top_flare_node_ids = [n for n, _ in flare_sorted[:TOP_N_FLARE_NODES]]

feat_profile_cols = [
    'monthlyrevenue', 'monthlyminutes', 'customercarecalls', 'droppedcalls',
    'blockedcalls', 'overageminutes', 'monthsinservice', 'percchangerevenues'
]
feat_profile_cols = [f for f in feat_profile_cols if f in X_imputed.columns]

profiles = []
for node in top_flare_node_ids:
    members = graph['nodes'][node]
    row = {'node': node,
           'churn_rate': f"{G.nodes[node]['churn_rate']:.1%}",
           'n_customers': len(members)}
    for f in feat_profile_cols:
        row[f] = round(X_imputed.iloc[members][f].mean(), 2)
    profiles.append(row)

# Add global baseline
baseline = {'node': '--- GLOBAL BASELINE ---', 'churn_rate': f"{y.mean():.1%}",
            'n_customers': len(df)}
for f in feat_profile_cols:
    baseline[f] = round(X_imputed[f].mean(), 2)
profiles.append(baseline)

profile_df = pd.DataFrame(profiles).set_index('node')
print("Feature profiles for top flare nodes vs. global baseline:")
print(profile_df.to_string())

---
## 📈 Stage 1E: Interactive Mapper Graph Visualization

In [ ]:
# Layout weighted by shared membership
pos = nx.spring_layout(G, weight='weight', seed=42, k=1.8)

# ── Edge traces ───────────────────────────────────────────────────────────────
edge_traces = []
for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]; x1, y1 = pos[v]
    alpha = 0.2 + d['weight'] * 0.5
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode='lines',
        line=dict(width=d['weight'] * 2.5, color=f'rgba(180,180,180,{alpha:.2f})'),
        hoverinfo='none', showlegend=False
    ))

# ── Node trace ────────────────────────────────────────────────────────────────
node_x, node_y, node_colors, node_sizes, node_texts = [], [], [], [], []

for node in G.nodes():
    nd  = G.nodes[node]
    x, y_pos = pos[node]
    node_x.append(x)
    node_y.append(y_pos)
    node_colors.append(nd.get('churn_rate', 0))
    node_sizes.append(max(6, min(45, nd.get('size', 1) ** 0.55)))

    # Hover text with key Cell2Cell metrics
    lines = [f"<b>Node:</b> {node}",
             f"<b>Customers:</b> {int(nd.get('size', 0)):,}",
             f"<b>Churn Rate:</b> {nd.get('churn_rate', 0):.1%}",
             f"<b>Avg Churn Prob:</b> {nd.get('churn_proba', 0):.1%}"]
    for feat in feat_profile_cols[:5]:
        k = f'mean_{feat}'
        if k in nd:
            lines.append(f"<b>{feat}:</b> {nd[k]:.1f}")
    node_texts.append('<br>'.join(lines))

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    text=node_texts,
    marker=dict(
        showscale=True,
        colorscale='RdYlGn_r',
        cmin=0, cmax=1,
        color=node_colors,
        size=node_sizes,
        colorbar=dict(
            thickness=15,
            title=dict(text='Churn Rate', side='right'),
            tickformat='.0%'
        ),
        line=dict(width=1.5, color='white')
    )
)

# ── Flare highlight ───────────────────────────────────────────────────────────
flare_trace = go.Scatter(
    x=[pos[n][0] for n in top_flare_node_ids if n in pos],
    y=[pos[n][1] for n in top_flare_node_ids if n in pos],
    mode='markers',
    name='🔴 Churn Flare Nodes',
    marker=dict(symbol='star', size=22, color='red',
                line=dict(color='darkred', width=2)),
    hoverinfo='text',
    text=[f"⭐ FLARE: {n}" for n in top_flare_node_ids if n in pos]
)

fig = go.Figure(
    data=edge_traces + [node_trace, flare_trace],
    layout=go.Layout(
        title=dict(
            text='<b>Cell2Cell — KeplerMapper Topological Graph</b><br>'
                 '<sup>Each node = a cluster of customers | Color = Churn Rate | ⭐ = Churn Flare</sup>',
            x=0.5
        ),
        showlegend=True,
        hovermode='closest',
        height=750,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        paper_bgcolor='#0f1117',
        plot_bgcolor='#0f1117',
        font=dict(color='white', size=12)
    )
)
fig.show()
print("\n🔴 Red arms/flares = concentrated churn neighborhoods")
print("🟢 Green core = stable, low-churn customers")
print("📌 Hover any node to see its feature profile")

In [ ]:
# ── Flare profile bar chart ───────────────────────────────────────────────────
# Compare flare mean vs global mean for key features

flare_member_indices = []
for node in top_flare_node_ids:
    flare_member_indices.extend(graph['nodes'].get(node, []))
flare_member_indices = list(set(flare_member_indices))

compare_feats = feat_profile_cols
flare_means  = X_imputed.iloc[flare_member_indices][compare_feats].mean()
global_means = X_imputed[compare_feats].mean()
ratios       = (flare_means / (global_means + 1e-9)).rename('ratio')

fig2, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#e74c3c' if r > 1.1 else '#2ecc71' if r < 0.9 else '#95a5a6'
              for r in ratios.values]
ratios.sort_values().plot(kind='barh', ax=ax, color=sorted(bar_colors))
ax.axvline(x=1.0, color='white', linestyle='--', linewidth=1.5, label='Global baseline')
ax.set_xlabel('Flare Mean / Global Mean  (ratio)')
ax.set_title('Churn Flare Feature Profile vs. Global Baseline\n'
             'Red bars = elevated in flare  |  Green = suppressed', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('flare_profile_cell2cell.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop elevated features in churn flare:")
print(ratios.sort_values(ascending=False).head(5).round(2))

---
## 🎯 Stage 2: Topological Sample Extraction

In [ ]:
# Collect all members from the top flare nodes
flare_sample_indices = list(set(flare_member_indices))
flare_sample_df = df.iloc[flare_sample_indices].copy()
flare_sample_df['churn_proba'] = churn_proba[flare_sample_indices]

# Sort by predicted churn probability — extreme tip = highest probability
flare_sample_df = flare_sample_df.sort_values('churn_proba', ascending=False)

# Take top 200 most extreme (enough for DoWhy estimation, not too large for LLM)
TOPO_SAMPLE_SIZE = 200
topo_sample = flare_sample_df.head(TOPO_SAMPLE_SIZE)

print("=" * 60)
print("TOPOLOGICAL SAMPLE — Cell2Cell Churn Flare")
print("=" * 60)
print(f"Total flare membership:   {len(flare_sample_df):,}")
print(f"Topological sample size:  {len(topo_sample)}")
print(f"Churn rate in sample:     {topo_sample[CHURN_COL].mean():.1%}")
print(f"Global dataset churn:     {y.mean():.1%}")
print(f"Enrichment factor:        {topo_sample[CHURN_COL].mean() / y.mean():.1f}x")
print()
print("Key feature means in topological sample vs. global:")
comparison_df = pd.DataFrame({
    'Flare Sample':  topo_sample[compare_feats].mean(),
    'Global Mean':   df[compare_feats].mean(),
}).round(2)
comparison_df['Ratio'] = (comparison_df['Flare Sample'] / comparison_df['Global Mean']).round(2)
print(comparison_df.sort_values('Ratio', ascending=False))

In [ ]:
# ── Save outputs for Stage 3 + 4 ─────────────────────────────────────────────
import json

# Topological sample with real Cell2Cell columns
topo_sample.to_csv('topo_flare_sample_cell2cell.csv', index=False)

# Full dataset with topology labels
df['churn_proba']  = churn_proba
df['is_in_flare']  = df.index.isin(flare_sample_indices)
df['topo_churn_enriched'] = (df['churn_proba'] > 0.6).astype(int)
df.to_csv('cell2cell_with_topology.csv', index=False)

# Metadata for LLM prompt construction in Stage 3
feature_descriptions = {
    'monthlyrevenue':        'Total monthly revenue billed to customer ($)',
    'monthlyminutes':        'Total call minutes used in the month',
    'customercarecalls':     'Number of inbound calls to customer care (dissatisfaction proxy)',
    'droppedcalls':          'Number of dropped calls (network quality failure)',
    'blockedcalls':          'Number of blocked calls (network congestion)',
    'overageminutes':        'Minutes used beyond plan limit (pricing friction)',
    'percchangerevenues':    'Month-over-month % change in revenue (engagement trend)',
    'percchangeminutes':     'Month-over-month % change in call minutes',
    'monthsinservice':       'Tenure in months (customer age)',
    'roamingcalls':          'Number of roaming calls (service area expansion)',
    'directorassistedcalls': 'Calls requiring directory assistance (complexity/friction)',
    'retentioncall':         'Whether customer received a retention call (1=Yes)',
    'retentionoffer':        'Whether customer received a retention offer (1=Yes)',
    'madecalltoretentionteam': 'Whether customer proactively called retention team (1=Yes)',
    'handsetprice':          'Price of customer handset ($)',
}

stage3_metadata = {
    'churn_column': CHURN_COL,
    'feature_cols': FEATURE_COLS,
    'behavioral_features': NUMERIC_BEHAVIORAL,
    'feature_descriptions': feature_descriptions,
    'flare_stats': {
        'n_flare_nodes': len(top_flare_node_ids),
        'total_flare_customers': len(flare_sample_df),
        'topo_sample_size': TOPO_SAMPLE_SIZE,
        'topo_sample_churn_rate': float(topo_sample[CHURN_COL].mean()),
        'global_churn_rate': float(y.mean()),
        'enrichment_factor': float(topo_sample[CHURN_COL].mean() / y.mean()),
        'elevated_features': ratios.sort_values(ascending=False).head(5).to_dict()
    }
}

with open('stage3_metadata.json', 'w') as f:
    json.dump(stage3_metadata, f, indent=2)

print("✅ Saved:")
print("  • topo_flare_sample_cell2cell.csv  → Stage 3: LLM causal discovery input")
print("  • cell2cell_with_topology.csv      → Full enriched dataset")
print("  • stage3_metadata.json            → Feature descriptions for LLM prompt")
print()
print("=" * 60)
print("PIPELINE STATUS: Stage 1 ✅  Stage 2 ✅  Stage 3 🔜  Stage 4 🔜")
print("=" * 60)
print()
print("➡️  NEXT: Open stage3_metadata.json and feed feature_descriptions")
print("   into pywhyllm ModelSuggester to get the DAG hypothesis.")
print("   Expected hypothesis: customercarecalls + droppedcalls → churn")
print("   Then verify with DoWhy on topo_flare_sample_cell2cell.csv")

---
## 🗺️ Pipeline Status

```
✅  Stage 1: Topological Mapping
       └── KeplerMapper on 71K × 58 real Cell2Cell features
       └── 2D lens: churn_probability + UMAP-1
       └── Churn flares detected and scored

✅  Stage 2: Topological Sampling
       └── 200 customers from flare tip (churn enrichment: Nx baseline)
       └── topo_flare_sample_cell2cell.csv exported

🔜  Stage 3: LLM Causal Discovery  (pywhyllm)
       └── Input: stage3_metadata.json (feature names + descriptions)
       └── ModelSuggester → DAG hypothesis
       └── Likely: droppedcalls → customercarecalls → churn
                  overageminutes → churn (separate path)

🔜  Stage 4: Statistical Estimation  (DoWhy)
       └── CausalModel(data=topo_sample, graph=llm_dag)
       └── identify_effect() + estimate_effect()
       └── Output: hard causal effect sizes per feature
```

### References
- Kiciman et al. (2023) — *Causal Reasoning and Large Language Models* · arXiv:2305.00050
- Singh, Mémoli, Carlsson (2007) — *Topological Methods for High Dimensional Data*
- Sharma & Kiciman (2020) — *DoWhy: A Python package for causal inference*
- Cell2Cell dataset — Duke Fuqua / WSDM KDD Cup 2003